In [1]:
import requests
from urllib.request import HTTPBasicAuthHandler

In [2]:
%cd data
# RSS for 20 podcasts
RSS = {
    "https://feeds.acast.com/public/shows/63f77218886da700110bf9f9",
    "https://smartpod.no/feed/misjonen",
    "https://feeds.acast.com/public/shows/5f883c4673174a1b3a21b64b",
    "https://podkast.nrk.no/program/norsken_svensken_og_dansken.rss",
    "https://podkast.nrk.no/program/burde_vaert_pensum.rss",
    "https://feeds.megaphone.fm/hubermanlab",
    "https://www.omnycontent.com/d/playlist/e73c998e-6e60-432f-8610-ae210140c5b1/a91018a4-ea4f-4130-bf55-ae270180c327/44710ecc-10bb-48d1-93c7-ae270180c33e/podcast.rss",
    "https://feeds.megaphone.fm/WWO8086402096",
    "https://feeds.simplecast.com/RV1USAfC",
    "https://feeds.megaphone.fm/pivot",
    "https://feeds.simplecast.com/54nAGcIl",
    "https://feeds.simplecast.com/82FI35Px",
    "https://lexfridman.com/feed/podcast/",
    "https://rss.acast.com/checksandbalance",
    "https://rss.acast.com/theeconomistmoneytalks",
    "https://rss.acast.com/theeconomistasks",
    "https://feeds.megaphone.fm/RM4031649020",
    "https://podcastfeeds.nbcnews.com/HL4TzgYC",
    "https://feeds.simplecast.com/dxZsm5kX",
    "https://feeds.simplecast.com/Y8lFbOT4",
}

/home/adamj/podannote/data


In [ ]:
# Add a single podcast to database
res = requests.post("http://127.0.0.1:8000/podcasts/", json={
"rss": "https://www.omnycontent.com/d/playlist/e73c998e-6e60-432f-8610-ae210140c5b1/2bee9419-43de-46ce-8996-af2a01167517/84cf551f-a33b-41d8-b112-af2a01167541/podcast.rss"
})
res

In [ ]:
# Add each podcast in RSS to database via API

for podcast in RSS:
  print(podcast)
  res = requests.post("http://127.0.0.1:8000/podcasts/", json={
    "rss": podcast
  })
  print(res.status_code)

In [4]:
# gather the podcast slug, episode guid, and audio file link for transcription
podcasts = requests.get("http://127.0.0.1:8000/podcasts/")
podcasts = podcasts.json()
audio_files = []
for podcast in podcasts:
    slug = podcast.get("slug")
    episodes = requests.get(f"http://127.0.0.1:8000/podcasts/{slug}/episodes/")
    episodes = episodes.json()
    for episode in episodes:
        audio_files.append({"slug":slug, "guid":episode.get("guid"), "audio_file":episode.get("audio_link"), "language":episode.get("language")})

In [ ]:
import os

# download podcast episodes and save as .wav files (needed for pyannote, irrelevant for whisper)
for episode in audio_files:
    link = episode.get("audio_file")
    guid = episode.get("guid")
    slug = episode.get("slug")
    filename = f"{slug}_{guid}"
    if not os.path.exists(filename + ".wav"):
        !curl -o '{filename}' -L -J '{link}'
        !ffmpeg -i '{filename}' -vn -acodec pcm_s16le -ar 16000 -ac 1 '{filename + ".wav"}'
        !rm '{filename}'
        print(f"Downloaded {filename}")


In [5]:
import whisper

In [6]:
# convert time in seconds to time in hours, minutes, seconds, 
#including leading zeros and rounding seconds
def convert_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = round(seconds % 60)
    return "{:02d}:{:02d}:{:02d}".format(hours, minutes, seconds)



In [7]:
whisper_details = !pip show openai-whisper
# find element in whisper_details that starts with "Version: "
whisper_version = [x.replace("Version: ", "") for x in whisper_details if x.startswith("Version: ")][0]
whisper_version

'20230314'

In [8]:
import datetime
import time

for episode in audio_files[90:]:
    guid = episode.get("guid")
    slug = episode.get("slug")
    filename = f"{slug}_{guid}.wav"

    model_size = "large"
    model = whisper.load_model(model_size)
    decode_options = dict(best_of=5, beam_size=5)
    transcribe_options = dict(word_timestamps=True, fp16=False, **decode_options)

    start_time = time.time()
    output = model.transcribe(filename, **transcribe_options)
    running_time = time.time() - start_time

    list_of_word_list = [seg_dict.pop("words") for seg_dict in output.get("segments")]
    words = [words for seglist in list_of_word_list for words in seglist]

    speech2txt = {
        "model": "OpenAI Whisper",
        "size": model_size,
        "version": whisper_version,
        "weights": whisper._MODELS.get(model_size),
        "hyperparams": transcribe_options,
    }

    transcription_dict = {
        'guid': guid,
        'name': f"Whisper-{model_size}",
        'runtime': convert_time(running_time),
        'created': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'text': output.get("text"),
        'language': output.get("language"),
        'speech2txt': speech2txt,
        'json': words
    }

    res = requests.post(f"http://127.0.0.1:8000/podcasts/{slug}/{guid}/transcriptions/", json=transcription_dict)

    print(slug, guid, convert_time(running_time))
    print(res)

    trans_uuid = res.json().get("uuid")

    segmentation_dict = {
        "uuid": trans_uuid,
        "name": "Whisper-default",
        "segmentor": speech2txt,
        "utterance_set": [utterance for utterance in output.get("segments") if utterance.get("text") != ""]
    }

    res = requests.post(f"http://127.0.0.1:8000/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict)

    print(res)




checks-and-balance-from-the-economist 640b6077aaa42400114ff7a8 00:09:57
<Response [201]>
<Response [201]>
checks-and-balance-from-the-economist 6402222fdf9463001123f22d 00:07:20
<Response [201]>
<Response [201]>
checks-and-balance-from-the-economist 63f8d8ad1a63070011203379 00:11:19
<Response [201]>
<Response [201]>
checks-and-balance-from-the-economist 63efc350cef8ec00116f58e5 00:07:44
<Response [201]>
<Response [201]>
checks-and-balance-from-the-economist 63e680b45c30a70010168ae3 00:07:52
<Response [201]>
<Response [201]>
misjonen-med-antonsen-og-golden f4dec7fe-b4d2-4344-8440-c256d6ca54a9 00:10:46
<Response [201]>
<Response [201]>
misjonen-med-antonsen-og-golden 5a7d0a1e-a2e7-49a0-a0bc-4a7233dc8d60 00:07:41
<Response [201]>
<Response [201]>
misjonen-med-antonsen-og-golden 3d6667a0-a038-4fd9-ab7f-9045ce81c3cb 00:11:26
<Response [201]>
<Response [201]>
misjonen-med-antonsen-og-golden 13625c6e-467e-4e39-97c9-a97d7fa0b035 00:12:53
<Response [201]>
<Response [201]>
misjonen-med-antonsen-